In [2]:
import h5pyd
import h5py
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import haversine
from haversine import haversine, Unit
import numpy as np
from datetime import datetime
import seaborn as sb

In [ ]:
#Not needed to run but handy to know that if you want to check what environment you're running python in you can do 
!conda env list  

In [3]:
sites = pd.read_csv('C:/Users/sarah/Documents/GitHub/ORKA_StatusReport_FollowupPaper/StatusReport_FollowupPaper_Data/MasterSiteList_FollowupPaper.csv')
sites = sites.iloc[0:15,0:8]

In [1]:
#shows daily salt data from 2008/10/01 - 2018/10/28 
#3680 unique time steps over 24039 unique lat/longs for 88463520 rows
#shows only surface salinity data
#restrict to 2013-2018 time
salt = pd.read_csv("D:/ORKA_StatusReportFollowupPaper/NOAA_all/surf_salt_all.csv")
salt = salt.dropna()
salt.reset_index(inplace=True)
salt.ocean_time.nunique()
len(salt[['lon_rho','lat_rho']].drop_duplicates())

NameError: name 'pd' is not defined

In [5]:
%%time
#Ok here I'm finding which ROMS grid cell is closest to each site 
#then pulling out the ROMS data for each nearest grid cell and putting it into a dataframe called 'salt_forsites'
lat = sites.SiteLatitude[0]
lon = sites.SiteLongitude[0]
site = (lat, lon)
salt_sub = salt.loc[(salt.lat_rho < site[0] +0.01) & (salt.lat_rho > site[0]-0.01)]
def hav_dist(row):
    site2 = (row["lat_rho"],row["lon_rho"])
    return haversine(site,site2)
dist = salt_sub.apply(hav_dist, axis = 1)
index1 = dist[dist == dist.min()].index
salt_forsites = salt.iloc[index1.values[:],]
print(site)

for i in range(1,len(sites)): 
    #print(i, "has the latitude", sites.SiteLatitude[i])
    site = (sites.SiteLatitude[i], sites.SiteLongitude[i])
    salt_sub = salt.loc[(salt.lat_rho < site[0] +0.01) & (salt.lat_rho > site[0]-0.01)]
    dist = salt_sub.apply(hav_dist, axis = 1)
    index1 = dist[dist == dist.min()].index
    closest = salt.iloc[index1.values[:],]
    salt_forsites = pd.concat([salt_forsites,closest])
    print(site)
    print(i," is finished, hooray!")


(np.float64(45.33673), np.float64(-123.98))
(np.float64(45.21247), np.float64(-123.985186))
1  is finished, hooray!
(np.float64(44.7844682), np.float64(-124.0742001))
2  is finished, hooray!
(np.float64(44.74663003), np.float64(-124.0725912))
3  is finished, hooray!
(np.float64(43.339313), np.float64(-124.378642))
4  is finished, hooray!
(np.float64(43.31574013), np.float64(-124.4044225))
5  is finished, hooray!
(np.float64(43.30083932), np.float64(-124.4007204))
6  is finished, hooray!
(np.float64(42.831791), np.float64(-124.581814))
7  is finished, hooray!
(np.float64(42.78849071), np.float64(-124.5953684))
8  is finished, hooray!
(np.float64(42.73575941), np.float64(-124.5077475))
9  is finished, hooray!
(np.float64(42.70231218), np.float64(-124.4692798))
10  is finished, hooray!
(np.float64(42.66761529), np.float64(-124.4408763))
11  is finished, hooray!
(np.float64(42.44372583), np.float64(-124.4683823))
12  is finished, hooray!
(np.float64(42.065901), np.float64(-124.31788))
13  

In [6]:
#double checking time is in correct data type and is restricted to 2013-2018
salt_forsites.ocean_time = pd.to_datetime(salt_forsites.ocean_time)
print(salt_forsites.ocean_time.min())
print(salt_forsites.ocean_time.max())
salt_forsites = salt_forsites.loc[salt_forsites.ocean_time >= "2013-01-01 12:00:00"].reset_index()
print(salt_forsites.ocean_time.min())
print(salt_forsites.ocean_time.max())

2008-10-01 12:00:00
2018-10-28 12:00:00
2013-01-01 12:00:00
2018-10-28 12:00:00


In [7]:
#Check which grid cells from ROMs model are pulled out to correspond with which sites
#Problem though, drake point and simpson are so clsoe that they end up pulling from a single cell. 
#So you'll need to give them both the same value when you assign to sites
#This is why you're only getting salinity metrics for 14 sites instead of 15
salt_forsites[["lat_rho","lon_rho"]].drop_duplicates()

,lat_rho,lon_rho
0,45.331876,-123.979773
2127,45.208695,-123.981258
4254,44.782938,-124.085730
6381,44.750297,-124.072604
8508,43.344360,-124.405358
10635,43.306547,-124.413856
14889,42.836038,-124.580172
17016,42.798020,-124.588188
19143,42.743022,-124.514910
21270,42.699168,-124.471149


In [8]:
#10th percentile of salinity values at each site
salt10th = salt_forsites.groupby("lat_rho").surf_salt.quantile(q = 0.1).reset_index()
salt10th
#need to insert a duplicate for the row with lat_rho of 43.306547, 
salt10th = pd.concat([salt10th.iloc[0:9,:],salt10th.iloc[8:14,:]]).reset_index()
thing = salt10th.iloc[:,2]
thing = thing.iloc[::-1]
sites["P10_salinity"] = thing.values
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_salinity
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,28.863858
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,27.747748
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,29.687819
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,29.778896
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,30.522499
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,30.566652
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,30.566652
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,30.980505
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,31.243691
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,30.077131


In [9]:
#Create function to get percentage of time that each site's salinity is less than 30 psu
def time_below_30psu(data):
    return len(data.loc[data.surf_salt < 30])/len(data)*100


In [10]:
#ok implementing that function for each site
saltunder30 = salt_forsites.groupby("lat_rho").apply(time_below_30psu).reset_index()
saltunder30 = pd.concat([saltunder30.iloc[0:9,:],saltunder30.iloc[8:14,:]]).reset_index()
saltunder30

C:\Users\sarah\AppData\Local\Temp\ipykernel_18728\3555101326.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  saltunder30 = salt_forsites.groupby("lat_rho").apply(time_below_30psu).reset_index()


,index,lat_rho,0
0,0,42.034555,22.849083
1,1,42.061861,18.570757
2,2,42.448852,30.794546
3,3,42.666432,11.330512
4,4,42.699168,10.860367
5,5,42.743022,9.449929
6,6,42.798020,2.491772
7,7,42.836038,3.244006
8,8,43.306547,4.795487
9,8,43.306547,4.795487


In [11]:
#Adding this new salinity metric for each site into a dataframe that shows multiple salinity metrics for each site
thing = saltunder30.iloc[:,2]
thing = thing.iloc[::-1]
sites["Values_Under_30psu"] = thing.values
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_salinity,Values_Under_30psu
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,28.863858,22.143865
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,27.747748,38.598966
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,29.687819,14.151387
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,29.778896,12.317819
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,30.522499,4.419370
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,30.566652,4.795487
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,30.566652,4.795487
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,30.980505,3.244006
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,31.243691,2.491772
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,30.077131,9.449929


In [12]:
#let's just look at growing season vs. winter and add these metrics into  dataframe that shows multiple salinity metrics for each site
#10th percentile of salinity values at each site
salt_growing = salt_forsites.loc[(salt_forsites.ocean_time.dt.month>=4) & (salt_forsites.ocean_time.dt.month<=9)]
salt10th_growing = salt_growing.groupby("lat_rho").surf_salt.quantile(q = 0.1).reset_index()
salt10th_growing
#need to insert a duplicate for the row with lat_rho of 43.306547, 
salt10th_growing = pd.concat([salt10th_growing.iloc[0:9,:],salt10th_growing.iloc[8:14,:]]).reset_index()
thing = salt10th_growing.iloc[:,2]
thing = thing.iloc[::-1]
sites["P10_growing"] = thing.values
sites
salt_winter = salt_forsites.loc[(salt_forsites.ocean_time.dt.month<=3) | (salt_forsites.ocean_time.dt.month>=10)]
salt10th_winter = salt_winter.groupby("lat_rho").surf_salt.quantile(q = 0.1).reset_index()
salt10th_winter
salt10th_winter = pd.concat([salt10th_winter.iloc[0:9,:],salt10th_winter.iloc[8:14,:]]).reset_index()
thing = salt10th_winter.iloc[:,2]
thing = thing.iloc[::-1]
sites["P10_winter"] = thing.values
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_salinity,Values_Under_30psu,P10_growing
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,28.863858,22.143865,27.943325
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,27.747748,38.598966,27.906660
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,29.687819,14.151387,29.716728
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,29.778896,12.317819,30.213950
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,30.522499,4.419370,31.065773
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,30.566652,4.795487,31.185372
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,30.566652,4.795487,31.185372
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,30.980505,3.244006,31.902587
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,31.243691,2.491772,31.964958
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,30.077131,9.449929,32.071026


In [14]:
#calculate percent of time under 30 psu for summer and winter
#and add these metrics into  dataframe that shows multiple salinity metrics for each site
saltunder30_growing = salt_growing.groupby("lat_rho").apply(time_below_30psu).reset_index()
saltunder30 = pd.concat([saltunder30_growing.iloc[0:9,:],saltunder30_growing.iloc[8:14,:]]).reset_index()
saltunder30_growing
saltunder30_growing = pd.concat([saltunder30_growing.iloc[0:9,:],saltunder30_growing.iloc[8:14,:]]).reset_index()
thing = saltunder30_growing.iloc[:,2]
thing = thing.iloc[::-1]
sites["Values_Under_30psu_growing"] = thing.values
sites

C:\Users\sarah\AppData\Local\Temp\ipykernel_18728\3110747039.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  saltunder30_growing = salt_growing.groupby("lat_rho").apply(time_below_30psu).reset_index()


,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_salinity,Values_Under_30psu,P10_growing,P10_winter,Values_Under_30psu_growing
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,28.863858,22.143865,27.943325,29.508668,25.318761
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,27.747748,38.598966,27.906660,27.695667,27.504554
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,29.687819,14.151387,29.716728,29.667985,12.021858
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,29.778896,12.317819,30.213950,29.623225,8.378871
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,30.522499,4.419370,31.065773,30.258514,2.185792
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,30.566652,4.795487,31.185372,30.128573,1.001821
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,30.566652,4.795487,31.185372,30.128573,1.001821
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,30.980505,3.244006,31.902587,30.399050,0.455373
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,31.243691,2.491772,31.964958,30.639451,0.364299
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,30.077131,9.449929,32.071026,29.402708,2.003643


In [15]:
#calculate percent of time under 30 psu for summer and winter
#and add these metrics into  dataframe that shows multiple salinity metrics for each site
saltunder30_winter = salt_winter.groupby("lat_rho").apply(time_below_30psu).reset_index()
saltunder30_winter = pd.concat([saltunder30_winter.iloc[0:9,:],saltunder30_winter.iloc[8:14,:]]).reset_index()
saltunder30_winter
thing = saltunder30_winter.iloc[:,2]
thing = thing.iloc[::-1]
sites["Values_Under_30psu_winter"] = thing.values
sites

C:\Users\sarah\AppData\Local\Temp\ipykernel_18728\1646544694.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  saltunder30_winter = salt_winter.groupby("lat_rho").apply(time_below_30psu).reset_index()


,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_salinity,Values_Under_30psu,P10_growing,P10_winter,Values_Under_30psu_growing,Values_Under_30psu_winter
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,28.863858,22.143865,27.943325,29.508668,25.318761,18.756074
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,27.747748,38.598966,27.906660,27.695667,27.504554,50.437318
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,29.687819,14.151387,29.716728,29.667985,12.021858,16.423712
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,29.778896,12.317819,30.213950,29.623225,8.378871,16.520894
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,30.522499,4.419370,31.065773,30.258514,2.185792,6.802721
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,30.566652,4.795487,31.185372,30.128573,1.001821,8.843537
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,30.566652,4.795487,31.185372,30.128573,1.001821,8.843537
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,30.980505,3.244006,31.902587,30.399050,0.455373,6.219631
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,31.243691,2.491772,31.964958,30.639451,0.364299,4.761905
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,30.077131,9.449929,32.071026,29.402708,2.003643,17.395530


In [16]:
sites.to_csv('C:/Users/sarah/Documents/GitHub/ORKA_StatusReport_FollowupPaper/StatusReport_FollowupPaper_Data/MasterSites_WithSalinityMetrics_20132018.csv')